# DAC IFEST 2026 - Title-Content Matching
## Pipeline: Preprocessing → Feature Extraction → Ensemble Model

In [ ]:
# Install dependencies
!pip install Sastrawi lightgbm -q

## 1. Upload Data
Upload `train.csv`, `test.csv`, dan `sample_submission.csv` ke folder `data/` di Colab.

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# Upload files manually via Colab file panel, atau jalankan cell ini:
from google.colab import files
print("Upload train.csv, test.csv, sample_submission.csv:")
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f'data/{fname}')
    print(f'  Moved {fname} -> data/{fname}')

## 2. Config & Imports

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
# Config
SEED = 42
MAX_FEATURES = 20000
NGRAM_RANGE = (1, 3)
VECTOR_SIZE = 100
WINDOW_SIZE = 5
N_FOLDS = 5
MAX_CONTENT_WORDS = 512
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(SEED)
print('Config loaded.')


## 3. Load Data & EDA

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample_sub = pd.read_csv('data/sample_submission.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
print(f'Sub:   {sample_sub.shape}')

# --- EDA & Cleaning ---
print(f'\nDuplikat di Train: {train.duplicated(subset=["title","content"]).sum()}')
train = train.drop_duplicates(subset=['title','content'], keep='first').reset_index(drop=True)
train = train.dropna(subset=['title','content','label']).reset_index(drop=True)
test['title'] = test['title'].fillna('').astype(str)
test['content'] = test['content'].fillna('').astype(str)

print(f'\nDistribusi Label:')
print(train['label'].value_counts())
print(f'\nTrain setelah cleaning: {train.shape}')

## 4. NLP Preprocessing (Sastrawi Stemming)

In [ ]:
stopwords_set = set(StopWordRemoverFactory().get_stop_words())
_stemmer = StemmerFactory().create_stemmer()
print('Stemmer initialized.')
def fast_clean(text, max_words=None):
    if not isinstance(text, str):
        text = str(text)
    if max_words:
        text = ' '.join(text.split()[:max_words])
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
print('1. Cleaning teks...')
train['temp_title'] = train['title'].apply(lambda x: fast_clean(x))
train['temp_content'] = train['content'].apply(lambda x: fast_clean(x, max_words=MAX_CONTENT_WORDS))
test['temp_title'] = test['title'].apply(lambda x: fast_clean(x))
test['temp_content'] = test['content'].apply(lambda x: fast_clean(x, max_words=MAX_CONTENT_WORDS))
# Word frequency
print('2. Hitung frekuensi kata...')
all_texts = train['temp_title'].tolist() + train['temp_content'].tolist() + test['temp_title'].tolist() + test['temp_content'].tolist()
word_counter = Counter(' '.join(all_texts).split())
print(f'   {len(word_counter)} kata unik ditemukan.')
# Vocab pruning
MIN_FREQ = 2
words_to_stem = []
stem_dict = {}
for word, freq in word_counter.items():
    if word in stopwords_set:
        continue
    if freq >= MIN_FREQ:
        words_to_stem.append(word)
    else:
        stem_dict[word] = word
print(f'3. {len(words_to_stem)} kata akan di-stem, {len(stem_dict)} kata langka dilewati.')


In [ ]:
# Stemming (single thread + cache - jauh lebih cepan)
def stem_with_cache(words, stemmer):
    cache = {}
    total = len(words)
    for i, word in enumerate(words):
        if word not in cache:
            cache[word] = stemmer.stem(word)
        if (i + 1) % 1000 == 0:
            print(f'   Stemming: {i+1}/{total} kata...')
    return cache
print(f'4. Stemming {len(words_to_stem)} kata (single thread + cache)...')
stem_dict = stem_with_cache(words_to_stem, _stemmer)
# Tambahkan kata langka
for word, freq in word_counter.items():
    if word in stopwords_set:
        continue
    if freq < MIN_FREQ:
        stem_dict[word] = word
def apply_stem_dict(text, stem_dict):
    return ' '.join(stem_dict[word] for word in text.split() if word in stem_dict)
print('5. Menerapkan stemming...')
train['clean_title'] = train['temp_title'].apply(lambda x: apply_stem_dict(x, stem_dict))
train['clean_content'] = train['temp_content'].apply(lambda x: apply_stem_dict(x, stem_dict))
test['clean_title'] = test['temp_title'].apply(lambda x: apply_stem_dict(x, stem_dict))
test['clean_content'] = test['temp_content'].apply(lambda x: apply_stem_dict(x, stem_dict))
train.drop(columns=['temp_title','temp_content'], inplace=True)
test.drop(columns=['temp_title','temp_content'], inplace=True)
print('Preprocessing selesai!')


## 5. Feature Extraction

In [ ]:
# ============================================
# 5a. TF-IDF (word + char)
# ============================================
print('Membuat fitur TF-IDF (word-level)...')
all_text = train['clean_title'].tolist() + train['clean_content'].tolist() + test['clean_title'].tolist() + test['clean_content'].tolist()

tfidf = TfidfVectorizer(ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES)
tfidf.fit(all_text)

train_title_tfidf = tfidf.transform(train['clean_title'])
train_content_tfidf = tfidf.transform(train['clean_content'])
test_title_tfidf = tfidf.transform(test['clean_title'])
test_content_tfidf = tfidf.transform(test['clean_content'])

idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

print('Membuat fitur TF-IDF (char-level)...')
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=5000)
tfidf_char.fit(all_text)

train_title_char = tfidf_char.transform(train['clean_title'])
train_content_char = tfidf_char.transform(train['clean_content'])
test_title_char = tfidf_char.transform(test['clean_title'])
test_content_char = tfidf_char.transform(test['clean_content'])

print(f'TF-IDF word features: {train_title_tfidf.shape[1]}')
print(f'TF-IDF char features: {train_title_char.shape[1]}')

In [ ]:
# ============================================
# 5b. Similarity Features
# ============================================
print('Menghitung fitur kemiripan...')

def compute_jaccard(s1, s2):
    if not s1 or not s2: return 0.0
    return len(s1 & s2) / len(s1 | s2)

def compute_dice(s1, s2):
    if not s1 or not s2: return 0.0
    return 2*len(s1 & s2) / (len(s1) + len(s2))

def compute_overlap_coeff(s1, s2):
    if not s1: return 0.0
    return len(s1 & s2) / min(len(s1), len(s2))

def compute_containment(s1, s2):
    if not s1: return 0.0
    return len(s1 & s2) / len(s1)

def row_wise_cosine_sim(mat_a, mat_b):
    dot = mat_a.multiply(mat_b).sum(axis=1)
    norm_a = np.sqrt(mat_a.multiply(mat_a).sum(axis=1))
    norm_b = np.sqrt(mat_b.multiply(mat_b).sum(axis=1))
    denom = np.asarray(norm_a).flatten() * np.asarray(norm_b).flatten()
    denom[denom == 0] = 1e-10
    return np.asarray(dot).flatten() / denom

for df, t_mat, c_mat, t_char, c_char in [
    (train, train_title_tfidf, train_content_tfidf, train_title_char, train_content_char),
    (test, test_title_tfidf, test_content_tfidf, test_title_char, test_content_char)
]:
    sets_t = [set(str(x).split()) for x in df['clean_title']]
    sets_c = [set(str(x).split()) for x in df['clean_content']]

    df['jaccard_sim'] = [compute_jaccard(t,c) for t,c in zip(sets_t, sets_c)]
    df['dice_sim'] = [compute_dice(t,c) for t,c in zip(sets_t, sets_c)]
    df['containment_sim'] = [compute_containment(t,c) for t,c in zip(sets_t, sets_c)]
    df['overlap_coeff'] = [compute_overlap_coeff(t,c) for t,c in zip(sets_t, sets_c)]
    df['tfidf_cosine_sim'] = row_wise_cosine_sim(t_mat, c_mat)
    df['char_tfidf_cosine_sim'] = row_wise_cosine_sim(t_char, c_char)

    t_len = df['clean_title'].apply(lambda x: len(str(x).split()))
    c_len = df['clean_content'].apply(lambda x: len(str(x).split()))
    df['title_len'] = t_len
    df['content_len'] = c_len
    df['length_ratio'] = c_len / (t_len + 1.0)
    df['length_diff'] = c_len - t_len

print('Fitur kemiripan selesai!')

In [ ]:
# ============================================
# 5c. Word2Vec (from scratch, no gensim)
# ============================================
print('Melatih Word2Vec dari scratch...')

class SimpleWord2Vec:
    def __init__(self, vector_size=100, window=5, min_count=2, seed=42):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.rng = np.random.RandomState(seed)
        self.word2idx = {}
        self.idx2word = {}
        self.W_in = None
        self.W_out = None

    def _build_vocab(self, sentences):
        word_freq = {}
        for sent in sentences:
            for w in sent:
                word_freq[w] = word_freq.get(w, 0) + 1
        idx = 0
        for w, c in word_freq.items():
            if c >= self.min_count:
                self.word2idx[w] = idx
                self.idx2word[idx] = w
                idx += 1
        self.vocab_size = len(self.word2idx)

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -8, 8)))

    def fit(self, sentences, epochs=5, neg_samples=5, lr_start=0.025, lr_min=0.0001):
        self._build_vocab(sentences)
        if self.vocab_size == 0: return self
        vs = self.vector_size
        self.W_in = (self.rng.rand(self.vocab_size, vs) - 0.5) / vs
        self.W_out = np.zeros((self.vocab_size, vs))
        total_words = sum(len(s) for s in sentences)
        for epoch in range(epochs):
            word_count = 0
            for sent in sentences:
                sent_indices = [self.word2idx[w] for w in sent if w in self.word2idx]
                for i, center_idx in enumerate(sent_indices):
                    word_count += 1
                    progress = word_count / (total_words * epochs)
                    lr = max(lr_min, lr_start * (1.0 - progress))
                    ctx_start = max(0, i - self.window)
                    ctx_end = min(len(sent_indices), i + self.window + 1)
                    for j in range(ctx_start, ctx_end):
                        if j == i: continue
                        ctx_idx = sent_indices[j]
                        dot = np.dot(self.W_in[center_idx], self.W_out[ctx_idx])
                        g = lr * (1.0 - self._sigmoid(dot))
                        self.W_out[ctx_idx] += g * self.W_in[center_idx]
                        self.W_in[center_idx] += g * self.W_out[ctx_idx]
                        for _ in range(neg_samples):
                            neg_idx = self.rng.randint(0, self.vocab_size)
                            dot = np.dot(self.W_in[center_idx], self.W_out[neg_idx])
                            g = lr * (0.0 - self._sigmoid(dot))
                            self.W_out[neg_idx] += g * self.W_in[center_idx]
                            self.W_in[center_idx] += g * self.W_out[neg_idx]
            print(f'  Word2Vec epoch {epoch+1}/{epochs} selesai')
        return self

    def __contains__(self, word): return word in self.word2idx
    def __getitem__(self, word): return self.W_in[self.word2idx[word]]

sentences = [str(text).split() for text in all_text]
w2v_model = SimpleWord2Vec(vector_size=VECTOR_SIZE, window=WINDOW_SIZE, min_count=2, seed=SEED)
w2v_model.fit(sentences, epochs=5, neg_samples=5)

def get_weighted_w2v(text):
    words = str(text).split()
    vecs, wts = [], []
    for w in words:
        if w in w2v_model:
            vecs.append(w2v_model[w])
            wts.append(idf_dict.get(w, 1.0))
    if not vecs: return np.zeros(VECTOR_SIZE)
    return np.average(vecs, axis=0, weights=wts)

for df in [train, test]:
    t_w2v = np.array([get_weighted_w2v(t) for t in df['clean_title']])
    c_w2v = np.array([get_weighted_w2v(t) for t in df['clean_content']])
    df['w2v_cosine_sim'] = [
        cosine_similarity(t.reshape(1,-1), c.reshape(1,-1))[0][0]
        if not (np.all(t==0) or np.all(c==0)) else 0.0
        for t, c in zip(t_w2v, c_w2v)
    ]

print('Word2Vec selesai!')

In [ ]:
# ============================================
# 5d. Hard Negative Features (Numbers, Entities, Negation)
# ============================================
print('Mengekstrak fitur angka, entitas, negasi...')

NUMBER_WORDS = {
    'satu':'1','dua':'2','tiga':'3','empat':'4','lima':'5',
    'enam':'6','tujuh':'7','delapan':'8','sembilan':'9','sepuluh':'10',
    'sebelas':'11','duabelas':'12','tigabelas':'13','empatbelas':'14',
    'limabelas':'15','enambelas':'16','tujuhbelas':'17','delapanbelas':'18',
    'sembilanbelas':'19','duapuluh':'20','tigapuluh':'30','empatpuluh':'40',
    'limapuluh':'50','enampuluh':'60','tujuhpuluh':'70','delapanpuluh':'80',
    'sembilanpuluh':'90','seratus':'100','ratus':'100','ribu':'1000',
    'juta':'1000000','miliar':'1000000000','pertama':'1','kedua':'2',
    'ketiga':'3','keempat':'4','kelima':'5'
}

NEGATION_WORDS = {
    'tidak','bukan','batal','gagal','tolak','menolak','bantah','membantah',
    'sangkal','menyangkal','tepis','menepis','hoaks','salah','klarifikasi',
    'keliru','palsu','bohong','dusta','fitnah','tipu','menipu'
}

HOAX_SIGNAL_WORDS = {
    'hoax','hoaks','palsu','bohong','fitnah','disinformasi','misinformasi',
    'salah','klarifikasi','bantahan','sanggahan','bukti','fakta','valid','terverifikasi'
}

def normalize_number_words(text):
    return ' '.join(NUMBER_WORDS.get(w, w) for w in str(text).lower().split())

def extract_numbers(text):
    return set(re.findall(r'\d+', normalize_number_words(text)))

def compute_number_overlap(title, content):
    nums_t = extract_numbers(title)
    nums_c = extract_numbers(content)
    if not nums_t: return -1.0
    return len(nums_t & nums_c) / len(nums_t)

def extract_capitalized_words(text):
    words = str(text).split()
    caps = set()
    for i, w in enumerate(words):
        cw = re.sub(r'[^A-Za-z]','',w)
        if len(cw)>1 and cw[0].isupper() and i!=0:
            caps.add(cw.lower())
    return caps

def compute_entity_overlap(title, content):
    caps_t = extract_capitalized_words(title)
    caps_c = extract_capitalized_words(content)
    if not caps_t: return -1.0
    return len(caps_t & caps_c) / len(caps_t)

def has_negation(text):
    return 1 if set(str(text).lower().split()) & NEGATION_WORDS else 0

def count_negation(text):
    return len(set(str(text).lower().split()) & NEGATION_WORDS)

def has_hoax_signal(text):
    return 1 if set(str(text).lower().split()) & HOAX_SIGNAL_WORDS else 0

def has_number(text):
    return 1 if re.search(r'\d+', str(text)) else 0

for df in [train, test]:
    df['number_overlap'] = df.apply(lambda x: compute_number_overlap(x['title'],x['content']), axis=1)
    df['entity_overlap'] = df.apply(lambda x: compute_entity_overlap(x['title'],x['content']), axis=1)
    df['title_negation'] = df['title'].apply(has_negation)
    df['content_negation'] = df['content'].apply(has_negation)
    df['negation_mismatch'] = (df['title_negation'] != df['content_negation']).astype(int)
    df['title_neg_count'] = df['title'].apply(count_negation)
    df['content_neg_count'] = df['content'].apply(count_negation)
    df['title_has_number'] = df['title'].apply(has_number)
    df['content_has_number'] = df['content'].apply(has_number)
    df['number_mismatch'] = (df['title_has_number'] != df['content_has_number']).astype(int)
    df['title_hoax_signal'] = df['title'].apply(has_hoax_signal)
    df['content_hoax_signal'] = df['content'].apply(has_hoax_signal)

print('Fitur hard negative selesai!')

In [ ]:
# ============================================
# 5e. Gabungkan semua fitur
# ============================================
extra_cols = [
    'jaccard_sim','dice_sim','containment_sim','overlap_coeff',
    'tfidf_cosine_sim','char_tfidf_cosine_sim','w2v_cosine_sim',
    'title_len','content_len','length_ratio','length_diff',
    'number_overlap','entity_overlap','negation_mismatch',
    'title_neg_count','content_neg_count','title_has_number',
    'content_has_number','number_mismatch','title_hoax_signal','content_hoax_signal',
]

X_train = sp.hstack([
    train_title_tfidf, train_content_tfidf,
    train_title_char, train_content_char,
    train[extra_cols].values
]).tocsr()

X_test = sp.hstack([
    test_title_tfidf, test_content_tfidf,
    test_title_char, test_content_char,
    test[extra_cols].values
]).tocsr()

y_train = train['label'].values

print(f'Total fitur: {X_train.shape[1]}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

## 6. Modelling (LightGBM + LogisticRegression Ensemble)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(X_train.shape[0])
test_lgb = np.zeros(X_test.shape[0])
oof_lr = np.zeros(X_train.shape[0])
test_lr = np.zeros(X_test.shape[0])

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'\n===== FOLD {fold+1} =====')
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]

    # LightGBM
    model_lgb = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, class_weight='balanced',
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_val,y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    val_lgb = model_lgb.predict_proba(X_val)[:,1]
    oof_lgb[val_idx] = val_lgb
    test_lgb += model_lgb.predict_proba(X_test)[:,1] / N_FOLDS
    print(f'  LightGBM | F1(@0.5): {f1_score(y_val, (val_lgb>=0.5).astype(int), average="macro"):.4f}')

    # LogisticRegression
    model_lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', solver='lbfgs', random_state=SEED)
    model_lr.fit(X_tr, y_tr)
    val_lr = model_lr.predict_proba(X_val)[:,1]
    oof_lr[val_idx] = val_lr
    test_lr += model_lr.predict_proba(X_test)[:,1] / N_FOLDS
    print(f'  LogReg   | F1(@0.5): {f1_score(y_val, (val_lr>=0.5).astype(int), average="macro"):.4f}')

# Cari bobot ensemble terbaik
print('\nCari bobot ensemble terbaik...')
best_w, best_f1 = 0.5, 0.0
for w in np.arange(0.0, 1.05, 0.05):
    combined = w * oof_lgb + (1-w) * oof_lr
    f1 = f1_score(y_train, (combined>=0.5).astype(int), average='macro')
    if f1 > best_f1:
        best_f1, best_w = f1, w

print(f'Bobot terbaik: LGB={best_w:.2f}, LR={1-best_w:.2f} | F1: {best_f1:.4f}')

oof_preds_probs = best_w * oof_lgb + (1-best_w) * oof_lr
test_preds_probs = best_w * test_lgb + (1-best_w) * test_lr

## 7. Post-Processing & Evaluation

In [ ]:
# Post-processing rule-based
print('Menerapkan post-processing...')

penalty_train = (
    ((train['title_has_number']==1) & (train['number_overlap']>=0) & (train['number_overlap']<0.4) & (train['tfidf_cosine_sim']<0.3)) |
    ((train['negation_mismatch']==1) & (train['tfidf_cosine_sim']<0.3))
)
oof_preds_probs[penalty_train] -= 0.2
oof_preds_probs = np.clip(oof_preds_probs, 0.0, 1.0)

penalty_test = (
    ((test['title_has_number']==1) & (test['number_overlap']>=0) & (test['number_overlap']<0.4) & (test['tfidf_cosine_sim']<0.3)) |
    ((test['negation_mismatch']==1) & (test['tfidf_cosine_sim']<0.3))
)
test_preds_probs[penalty_test] -= 0.2
test_preds_probs = np.clip(test_preds_probs, 0.0, 1.0)

# Threshold tuning
print('\n=== EVALUASI ===')
auc = roc_auc_score(y_train, oof_preds_probs)
print(f'ROC-AUC: {auc:.4f}')

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.05, 0.96, 0.01):
    f1 = f1_score(y_train, (oof_preds_probs >= th).astype(int), average='macro')
    if f1 > best_f1:
        best_f1, best_th = f1, th

print(f'Threshold optimal: {best_th:.2f}')
print(f'Macro F1: {best_f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_train, (oof_preds_probs>=best_th).astype(int), target_names=['Tidak Sesuai (0)','Sesuai (1)']))
print('Confusion Matrix:')
print(confusion_matrix(y_train, (oof_preds_probs>=best_th).astype(int)))

## 8. Generate Submission

In [ ]:
test_labels = (test_preds_probs >= best_th).astype(int)
sample_sub['label'] = test_labels
sample_sub.to_csv('submission.csv', index=False)

print(f'Submission saved! Threshold={best_th:.2f}, F1={best_f1:.4f}')
print(f'Distribusi prediksi:')
print(sample_sub['label'].value_counts())

# Download file
from google.colab import files
files.download('submission.csv')